# 006 Run a Full Weather Skill Session

这是第六课：做一次完整天气查询会话演练。

学习目标：

1. 理解一份 Skill 在真实使用时如何从“用户问题”一步步走到“可执行查询”
2. 学会把 workflow、references、scripts 放进一次完整会话里看
3. 理解“需要澄清”和“可以直接执行”这两条路径
4. 用真实天气 Skill 做一次端到端的小演练

这节课继续使用：

- `.agents/skills/weather-query-assistant/`


## 先明确这节课解决什么问题

到第五课为止，这份天气 Skill 已经有了：

- workflow
- references
- scripts
- mini workflow

但这些内容还是按模块分别看的。

现在我们要回答一个更接近真实使用的问题：

“如果用户真的发来一句天气问题，这份 Skill 会怎么一步步工作？”


## 先看当前真实目录

这节课不再加新目录，直接使用现有真实 Skill。


In [1]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


In [2]:
def print_tree(root: Path, prefix: str = '') -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for index, entry in enumerate(entries):
        connector = '└── ' if index == len(entries) - 1 else '├── '
        print(prefix + connector + entry.name)
        if entry.is_dir():
            next_prefix = prefix + ('    ' if index == len(entries) - 1 else '│   ')
            print_tree(entry, next_prefix)


print(skill_root)
print_tree(skill_root)


.agents/skills/weather-query-assistant
├── agents
│   └── openai.yaml
├── references
│   └── weather_sources.md
├── scripts
│   ├── __pycache__
│   │   ├── normalize_location.cpython-310.pyc
│   │   └── normalize_location.cpython-313.pyc
│   ├── build_wttr_query.py
│   └── normalize_location.py
└── SKILL.md


## 先读一次 `SKILL.md`

这一步不是复习，而是为了把整个会话流程重新对齐。

你应该能从这里读出两条分支：

1. 地点不明确 -> 先澄清
2. 地点明确 -> 标准化、构建查询、准备回答


In [3]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Normalize the location when the user input is noisy or inconsistently formatted.
4. Build a stable weather query string before calling the external service.
5. Use `wttr.in` as the primary source.
6. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
7. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
8. Do not guess when weather data is unavailable.

## References

- Read `referenc

## 这节课的边界

这节课重点是 Skill workflow，不是自然语言理解系统本身。

所以我们这里不追求做一个复杂的地点抽取器。

为了教学清晰，我们会把演练拆成两类输入：

1. `location_known = False`
2. `location_known = True`

也就是说，这一课先聚焦：

- 当地点不明确时，Skill 怎么处理
- 当地点明确时，Skill 怎么接脚本继续往下走


In [4]:
session_boundary = {
    'in_scope': ['clarify vs proceed', 'normalize location', 'build query URL', 'prepare final command'],
    'out_of_scope': ['complex NLU', 'real weather fetching', 'long-form travel advice'],
}

from pprint import pprint
pprint(session_boundary)


{'in_scope': ['clarify vs proceed',
              'normalize location',
              'build query URL',
              'prepare final command'],
 'out_of_scope': ['complex NLU',
                  'real weather fetching',
                  'long-form travel advice']}


## 定义一个最小会话执行函数

这里不做复杂 Agent，只做一个教学版的 session runner。

逻辑很简单：

1. 如果地点不明确，返回澄清问题
2. 如果地点明确，调用脚本生成查询 URL
3. 再拼出最终可执行命令


In [5]:
import subprocess


def build_weather_command(location: str, mode: str = 'compact') -> str:
    result = subprocess.run(
        [
            'python',
            '.agents/skills/weather-query-assistant/scripts/build_wttr_query.py',
            location,
            '--mode',
            mode,
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or 'build_wttr_query failed')
    return f'curl -s "{result.stdout.strip()}"'


def run_weather_session(user_question: str, location: str | None = None, mode: str = 'compact') -> dict:
    if not location:
        return {
            'user_question': user_question,
            'action': 'clarify',
            'assistant_reply': '请告诉我你要查询哪个城市或地点的天气。',
        }

    command = build_weather_command(location, mode=mode)
    return {
        'user_question': user_question,
        'action': 'proceed',
        'normalized_location_used': location,
        'command': command,
        'assistant_reply': '地点已明确，可以继续查询天气。',
    }


## 先看“需要澄清”的路径

这是第一条真实路径。

如果用户只说：

- “今天会下雨吗？”

那这份 Skill 不应该硬猜地点。


In [6]:
clarify_case = run_weather_session('今天会下雨吗？')
pprint(clarify_case)


{'action': 'clarify',
 'assistant_reply': '请告诉我你要查询哪个城市或地点的天气。',
 'user_question': '今天会下雨吗？'}


## 再看“可以直接执行”的路径

如果用户已经给了地点，例如：

- “帮我看下北京今天的天气”

那 Skill 就应该继续往下走，而不是再追问。


In [7]:
proceed_case = run_weather_session('帮我看下北京今天的天气', location='beijing', mode='current')
pprint(proceed_case)


{'action': 'proceed',
 'assistant_reply': '地点已明确，可以继续查询天气。',
 'command': 'curl -s "https://wttr.in/Beijing?m&format=3"',
 'normalized_location_used': 'beijing',
 'user_question': '帮我看下北京今天的天气'}


## 观察这条路径里真实发生了什么

这次不是一句话带过，而是清楚拆成几步：

1. `run_weather_session()` 判断地点已经明确
2. 调用 `build_wttr_query.py`
3. `build_wttr_query.py` 内部又会调用 `normalize_location()`
4. 返回最终的 `wttr.in` URL
5. 外层再拼成可执行 `curl` 命令

这就是一条完整的小执行链。


In [8]:
execution_chain = [
    'user question arrives',
    'skill decides location is clear',
    'build_wttr_query.py is called',
    'normalize_location() runs inside the script',
    'query-ready URL is returned',
    'final curl command is prepared',
]

pprint(execution_chain)


['user question arrives',
 'skill decides location is clear',
 'build_wttr_query.py is called',
 'normalize_location() runs inside the script',
 'query-ready URL is returned',
 'final curl command is prepared']


## 再试两个不同输入

这里的重点不是天气本身，而是看 workflow 有没有稳定工作。


In [9]:
examples = [
    {'question': '纽约现在什么天气？', 'location': ' New   York ', 'mode': 'current'},
    {'question': 'JFK 当前天气怎么样？', 'location': 'jfk', 'mode': 'compact'},
]

for item in examples:
    result = run_weather_session(item['question'], location=item['location'], mode=item['mode'])
    print(result)
    print('-' * 60)


{'user_question': '纽约现在什么天气？', 'action': 'proceed', 'normalized_location_used': ' New   York ', 'command': 'curl -s "https://wttr.in/New+York?m&format=3"', 'assistant_reply': '地点已明确，可以继续查询天气。'}
------------------------------------------------------------
{'user_question': 'JFK 当前天气怎么样？', 'action': 'proceed', 'normalized_location_used': 'jfk', 'command': 'curl -s "https://wttr.in/JFK?m&format=%l:+%c+%t+%h+%w"', 'assistant_reply': '地点已明确，可以继续查询天气。'}
------------------------------------------------------------


## 这节课最重要的认知变化

前几课你看到的是：

- 目录
- 文件
- 脚本
- references

这一课开始，你应该把它们合成一件事来看：

- 这份 Skill 已经能支撑一次完整的小会话

也就是说，现在它不只是“有结构”，而是真的开始有运行感了。


In [ ]:
what_changed = {
    'before': 'separate parts',
    'now': 'a connected session workflow',
}

pprint(what_changed)


## 真实开发里，这节课意味着什么

这一步的价值在于：

- 你已经能验证 Skill 是否真的可用
- 你不再只是讨论目录设计
- 你开始能从用户问题一路走到实际命令

这才是 Skill 从“结构化知识”走向“可操作工作流”的关键一步。


## 如果继续往下走，第七课最自然的方向是什么

现在最自然的下一步不是继续堆更多脚本，而是开始做“健壮性”。

例如第七课可以做：

1. 输入异常地点时怎么办
2. `build_wttr_query.py` 失败时怎么处理
3. 什么情况下该改走 Open-Meteo fallback

也就是说，下一课适合进入错误处理和 fallback 分支。


In [10]:
lesson_seven_candidate = {
    'theme': 'error handling and fallback paths',
    'topics': [
        'missing or noisy location',
        'script failure handling',
        'when to choose Open-Meteo fallback',
    ],
}

pprint(lesson_seven_candidate)


{'theme': 'error handling and fallback paths',
 'topics': ['missing or noisy location',
            'script failure handling',
            'when to choose Open-Meteo fallback']}


## 当前阶段结论

你现在需要记住：

1. 第六课的重点不是新增文件，而是把现有部分真正串成一次完整会话
2. 一份 Skill 只有当它能从用户问题一路走到可执行动作时，才真正开始有“运行感”
3. “需要澄清”和“可以继续执行”是两条同样重要的真实路径
4. 这份天气 Skill 现在已经能支撑从地点判断到查询命令生成的完整小会话
5. 下一步最自然就是补错误处理和 fallback 分支

下一步建议：

- 继续第七课：给天气 Skill 加入错误处理和 fallback 教学
